# Compare and select the model

## Selection rule

This comparison records the selection rule before displaying the scores: **select the baseline model with the highest mean grouped-validation Top-1 accuracy** (`cv_top1`). The score is the arithmetic mean of Top-1 accuracy across the three trip-grouped validation folds. Student-test metrics are reported only after selecting the model; they do not determine the winner. If validation Top-1 is exactly tied, use higher grouped-validation macro F1; if still tied, retain the order in the five-model list below.

The five baseline models are Logistic Regression, Gaussian Naive Bayes, Decision Tree, Random Forest, and HistGradientBoosting. The tuned HistGradientBoosting result is excluded from this five-model baseline comparison.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Locate the repository even when Jupyter starts in a child directory.
ROOT = next(
    (p for start in (Path.cwd(), *Path.cwd().parents)
     for p in (start, start / "pathsnap")
     if (p / "results/models.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate results/models.csv from the notebook's current directory.")

results = pd.read_csv(ROOT / "results/models.csv")
folds = pd.read_csv(ROOT / "results/review2_fold_summary.csv")

baseline_names = ["Logistic", "GaussianNB", "DecisionTree", "RandomForest", "HistBoost"]
baseline = results.set_index("model").loc[baseline_names].reset_index()
assert len(baseline) == 5, "Expected exactly five baseline models."
assert len(folds) == 3, "Expected three grouped-validation folds."
assert (folds["overlapping_trip_ids"] == 0).all(), "A trip overlaps between fit and validation."

# Selection uses only mean grouped-validation Top-1.
ranked = baseline.sort_values(
    ["cv_top1", "cv_macro_f1"], ascending=[False, False], kind="stable"
).reset_index(drop=True)
selected = ranked.iloc[0]
runner_up = ranked.iloc[1]
cv_gap = selected["cv_top1"] - runner_up["cv_top1"]

comparison = ranked[[
    "model", "cv_top1", "top1", "macro_f1", "log_loss", "inference_ms_per_obs", "accuracy"
]].rename(columns={
    "model": "Model",
    "cv_top1": "Mean grouped-validation Top-1",
    "top1": "Student-test Top-1",
    "macro_f1": "Student-test macro F1",
    "log_loss": "Student-test log loss",
    "inference_ms_per_obs": "Inference (ms/observation)",
    "accuracy": "Student-test candidate accuracy",
})
comparison.insert(1, "Selection", ["SELECTED" if name == selected["model"] else "" for name in comparison["Model"]])
display(comparison.style.format({
    "Mean grouped-validation Top-1": "{:.4f}",
    "Student-test Top-1": "{:.4f}",
    "Student-test macro F1": "{:.4f}",
    "Student-test log loss": "{:.4f}",
    "Inference (ms/observation)": "{:.6f}",
    "Student-test candidate accuracy": "{:.4f}",
}))

print(f"Selected model: {selected['model']}")
print(f"Runner-up: {runner_up['model']}")
print(f"Mean grouped-validation Top-1 gap: {cv_gap:.6f} ({cv_gap * 100:.3f} percentage points)")
print(
    f"Student-test: Top-1={selected['top1']:.4f}, "
    f"macro F1={selected['macro_f1']:.4f}, "
    f"log loss={selected['log_loss']:.4f}, "
    f"inference={selected['inference_ms_per_obs']:.6f} ms/observation."
)
print(
    f"Runner-up student-test Top-1={runner_up['top1']:.4f}, "
    f"macro F1={runner_up['macro_f1']:.4f}, "
    f"log loss={runner_up['log_loss']:.4f}, "
    f"inference={runner_up['inference_ms_per_obs']:.6f} ms/observation."
)

,Model,Selection,Mean grouped-validation Top-1,Student-test Top-1,Student-test macro F1,Student-test log loss,Inference (ms/observation),Student-test candidate accuracy
0,HistBoost,SELECTED,0.7770,0.8722,0.9251,0.0805,0.008546,0.9674
1,RandomForest,,0.7759,0.8715,0.9238,0.0816,0.015854,0.9674
2,DecisionTree,,0.7686,0.8685,0.9213,0.0988,0.000536,0.9663
3,Logistic,,0.7629,0.8525,0.8996,0.1211,0.000576,0.9567
4,GaussianNB,,0.7594,0.8442,0.8092,0.5102,0.001593,0.8930


Selected model: HistBoost
Runner-up: RandomForest
Mean grouped-validation Top-1 gap: 0.001070 (0.107 percentage points)
Student-test: Top-1=0.8722, macro F1=0.9251, log loss=0.0805, inference=0.008546 ms/observation.
Runner-up student-test Top-1=0.8715, macro F1=0.9238, log loss=0.0816, inference=0.015854 ms/observation.


## Interpretation and metric definitions

- **Selected model:** HistGradientBoosting has the highest mean grouped-validation Top-1 among the five baselines. Random Forest is the runner-up. The validation gap is about 0.107 percentage points, so the two are close.
- **Student-test performance:** HistGradientBoosting scores Top-1 0.8722, macro F1 0.9251, log loss 0.0805, and 0.008546 ms per observation for classifier inference. Random Forest scores Top-1 0.8715, macro F1 0.9238, log loss 0.0816, and 0.015854 ms per observation.
- **Top-1:** for each GPS observation, choose its candidate with the largest predicted probability and check whether it matches the reference arc.
- **Macro F1:** average of positive-class and negative-class F1 for candidate classification on the student-test set.
- **Log loss:** candidate-level probability loss on the student-test set; lower is better.
- **Inference time:** time spent in classifier `predict_proba`, divided by the number of observations. It excludes preprocessing, model loading, and HMM decoding.

The `student_test` split is a development comparison set with available reference routes, not a final hidden evaluator set. The source metrics are `results/models.csv`; fold trip counts and the no-overlap check are in `results/review2_fold_summary.csv`.